In [1]:
from delta import *
from pyspark.sql import *
from pyspark.sql.functions import *

get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

builder = (SparkSession.builder
           .appName("create-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-49c6b325-36f8-4fb2-ba75-6dcf229d4a3d;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 113ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0  

In [2]:
df = spark.read.format("delta").load("/opt/workspace/data/delta_lake/netflix_titles")

df.show(5)

25/05/31 10:15:41 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                null|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|               |Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         null|Septem

In [3]:
%%sparksql

ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` ALTER COLUMN title DROP NOT NULL

In [4]:
%%sparksql

ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` ADD CONSTRAINT validType CHECK (type IN ('Movie', 'Show', 'TV Show'));

In [8]:
%%sparksql

CREATE OR REPLACE TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles_shallow_clone` SHALLOW CLONE delta.`/opt/workspace/data/delta_lake/netflix_titles`

source_table_size,source_num_of_files,num_removed_files,num_copied_files,removed_files_size,copied_files_size
1963749,1,0,0,0,0


In [9]:
%%sparksql

ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` ADD COLUMNS (id INT);

In [10]:
%%sparksql
ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` ALTER COLUMN country AFTER show_id;

In [12]:
%%sparksql

ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` SET TBLPROPERTIES (
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5',
    'delta.columnMapping.mode' = 'name'
)

In [13]:
%%sparksql

ALTER TABLE delta.`/opt/workspace/data/delta_lake/netflix_titles` DROP COLUMN id;

In [14]:
spark.stop()